# Kámárí · 02 · Cleaning & Preprocessing  (Google Colab)

Turn raw datasets into one canonical manifest + aligned 224×224 crops.

Per image: **adapter → face detect → eye-align → crop 224 → quality score → ITA/skin band → manifest row**. Low-quality / no-face images are dropped from training.

**Use a GPU runtime** (Runtime → Change runtime type → GPU). Writes locally only — upload happens once in `04_push_to_huggingface.ipynb`.

In [ ]:
# --- Kámárí bootstrap: works on Google Colab and locally ---
import os, sys, pathlib
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    try:
        from google.colab import userdata
        for _k in ['HF_TOKEN','HF_NAMESPACE','KAGGLE_USERNAME','KAGGLE_KEY','FAGE_HF_REPO','KAMARI_REPO_URL']:
            try: os.environ.setdefault(_k, userdata.get(_k))
            except Exception: pass
    except Exception: pass
    if not pathlib.Path('/content/kamari/data').exists():
        os.system(f"git clone {os.environ.get('KAMARI_REPO_URL','')} /content/kamari")
    os.system('pip install -q -r /content/kamari/requirements-data.txt')
    REPO = pathlib.Path('/content/kamari')
else:
    REPO = next((c for c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
                 if (c/'data'/'registry').exists()), pathlib.Path.cwd().parent)
    from dotenv import load_dotenv; load_dotenv(REPO/'.env')
sys.path.append(str(REPO))
print('REPO =', REPO, '| Colab:', IN_COLAB)

In [ ]:
import numpy as np, pandas as pd, torch
from PIL import Image
from tqdm.auto import tqdm
from data.adapters import REGISTERED
from data import face_utils as F
from data.manifest_schema import empty_manifest, MANIFEST_COLUMNS, ita_to_band, validate

RAW = REPO / 'data' / 'raw'
CROPS = REPO / 'data' / 'processed' / 'crops_224'
for p in (CROPS, REPO/'data'/'manifests'): p.mkdir(parents=True, exist_ok=True)
QUALITY_MIN = 0.30
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| adapters:', list(REGISTERED))

In [ ]:
# 1) Collect raw rows from every adapter whose data is present
rows = []
for name, mod in REGISTERED.items():
    root = RAW / name.lower()
    if not root.exists():
        print('skip (not downloaded):', name); continue
    n = 0
    for r in mod.iter_rows(str(root)):
        rows.append(r); n += 1
    print(f'{name}: {n} raw rows')
print('total raw rows:', len(rows))

In [ ]:
# 2) Align, crop, score, skin band
def process(row):
    fp = pathlib.Path(row['path'])
    if not fp.exists(): return None
    try: rgb = np.array(Image.open(fp).convert('RGB'))
    except Exception: return None
    boxes, probs, lms = F.detect_faces(rgb, device=DEVICE)
    if boxes is None or len(boxes) == 0: return None
    i = int(np.argmax(probs))
    crop = F.align_and_crop(rgb, lms[i], size=224)
    ita = F.estimate_ita(crop); b = boxes[i]
    row.update(dict(face_quality=F.quality_score(crop), skin_ita=ita,
                    skin_band=ita_to_band(ita),
                    bbox=f"{int(b[0])},{int(b[1])},{int(b[2]-b[0])},{int(b[3]-b[1])}"))
    out = CROPS / f"{row['image_id'].replace('/', '__')}.jpg"
    Image.fromarray(crop).save(out, quality=95)
    row['path'] = str(out.relative_to(REPO))
    return row

kept = []
for r in tqdm(rows):
    pr = process(r)
    if pr and (pr.get('face_quality') or 0) >= QUALITY_MIN:
        kept.append(pr)
print(f'kept {len(kept)} / {len(rows)} after detect + quality>={QUALITY_MIN}')

In [ ]:
# 3) Build + validate manifest, derive splits, save three parquets
df = pd.concat([empty_manifest(), pd.DataFrame(kept)], ignore_index=True)[MANIFEST_COLUMNS]
df.loc[df['eval_ok'] == True, 'split'] = 'benchmark'
print('validation:', validate(df) or 'OK')

MAN = REPO / 'data' / 'manifests'
public = df.copy()
train = df[(df['train_ok'] == True) & (df['split'] != 'benchmark')].copy()
bench = df[df['split'] == 'benchmark'].copy()
for d, n in [(public,'public'), (train,'train'), (bench,'benchmark')]:
    d.to_parquet(MAN / f'manifest_{n}_v0.parquet', index=False)
print('saved rows:', {'public':len(public),'train':len(train),'benchmark':len(bench)})
print('\nNext: run 03_eda.ipynb')